## 1. Synthetic Urology Tests

Testing the Readability Agent on five sample urology explanations to see if it can simplify medical text to an 8th-grade reading level or lower. Comparing readability scores before and after rewriting.

In [20]:
import sys
import re
import pandas as pd
from pathlib import Path
from google import genai
from readability_agent import process_readability

In [7]:
notebooks_dir = (
    Path.cwd() / "notebooks"
    if(Path.cwd() / "notebooks").exists()
    else Path.cwd()
)
sys.path.insert(0, str(notebooks_dir))

pd.set_option("display.max_colwidth", None)

client = genai.Client()

In [8]:
# These are synthetic test cases, not quotations from MTSamples.
# We are using these test cases to test the readability agent's
# behavior accross five urology topics. 

test_cases = [
    {
        "condition": "Urinary Tract Infection",
        "text": (
            "The patient presents with dysuria, urinary frequency, "
            "and urgency, consistent with a urinary tract infection."
        ),
    },
    {
        "condition": "Kidney Stones",
        "text": (
            "Imaging demonstrates a 5 mm calculus in the left ureter, "
            "associated with mild hydronephrosis."
        ),
    },
    {
        "condition": "Hematuria",
        "text": (
            "The patient exhibits gross hematuria and requires "
            "further urological evaluation to determine its etiology."
        ),
    },
    {
        "condition": "Urinary Retention",
        "text": (
            "The patient experiences acute urinary retention "
            "requiring bladder decompression via catheterization."
        ),
    },
    {
        "condition": "Cystoscopy",
        "text": (
            "A cystoscopic examination was performed to "
            "evaluate the bladder mucosa for abnormalities."
        ),
    },
]

In [9]:
results = []

for case in test_cases:
    try:
        result = process_readability(case["text"], client)

        results.append({
            "Condition": case["condition"],
            "Original Text": result["original_text"],
            "Simplified Text": result["final_text"],
            "Original Grade": round(
                result["original_scores"]["flesch_kincaid_grade"], 2
            ),
            "Final Grade": round(
                result["final_scores"]["flesch_kincaid_grade"], 2
            ),
            "Original SMOG": round(
                result["original_scores"]["smog_index"], 2
            ),
            "Final SMOG": round(
                result["final_scores"]["smog_index"], 2
            ),
            "Meets Target": result["meets_target"],
            "Rewrites": result["rewrite_attempts"],
            "Needs Fact Check": result["requires_faithfulness_check"],
            "Error": None,
        })
    except Exception as error:
        results.append({
            "Condition": case["condition"],
            "Error": str(error),
            "Meets Target": False,
        })

df_results = pd.DataFrame(results)
display(df_results)

,Condition,Original Text,Simplified Text,Original Grade,Final Grade,Original SMOG,Final SMOG,Meets Target,Rewrites,Needs Fact Check,Error
0,Urinary Tract Infection,"The patient presents with dysuria, urinary frequency, and urgency, consistent with a urinary tract infection.","The patient has pain when urinating, needs to go often, and feels a strong, sudden need to go. These signs point to a urinary tract infection.",15.43,6.27,17.12,10.13,True,1,True,None
1,Kidney Stones,"Imaging demonstrates a 5 mm calculus in the left ureter, associated with mild hydronephrosis.",Imaging shows a 5 mm kidney stone in the left ureter. This is causing a mild backup of urine in the kidney.,11.78,4.25,15.90,7.17,True,1,True,None
2,Hematuria,The patient exhibits gross hematuria and requires further urological evaluation to determine its etiology.,The patient has blood in the urine that can be seen. The patient needs more tests from a urinary specialist to find the cause.,18.53,4.82,17.12,8.84,True,2,True,None
3,Urinary Retention,The patient experiences acute urinary retention requiring bladder decompression via catheterization.,"The patient cannot empty their bladder. A thin, flexible tube is placed to drain the urine.",24.10,4.49,17.12,7.17,True,1,True,None
4,Cystoscopy,A cystoscopic examination was performed to evaluate the bladder mucosa for abnormalities.,"A doctor used a thin, lighted tube to look inside the bladder to check for problems.",18.59,6.14,15.90,3.13,True,1,True,None


In [ ]:
successful = df_results[df_results["Error"].isna()].copy()

if not successful.empty:
    pass_rate = successful["Meets Target"].mean() * 100
    avg_before = successful["Original Grade"].mean()
    avg_after = successful["Final Grade"].mean()

    print(f"Successful tests: {len(successful)}/{len(test_cases)}")
    print(f"Readability pass rate: {pass_rate:.1f}%")
    print(f"Average grade before: {avg_before:.2f}")
    print(f"Average grade after: {avg_after:.2f}")
    print(f"Average improvement: {avg_before - avg_after:.2f}")

    display(successful[
        [
            "Condition",
            "Original Text",
            "Simplified Text",
            "Original Grade",
            "Final Grade",
            "Meets Target",
            "Rewrites",
            "Needs Fact Check"
        ]
    ])

    print("\nCASES THAT DID NOT MEET TARGET:")
    failed_target = successful[~successful["Meets Target"].astype(bool)]

    if failed_target.empty:
        print("All cases met the 8th-grade target!")
    else:
        display(failed_target[
            ["Condition", "Simplified Text", "Final Grade", "Rewrites"]
        ])


Successful tests: 5/5
Readability pass rate: 100.0%
Average grade before: 17.69
Average grade after: 5.19
Average improvement: 12.49


,Condition,Original Text,Simplified Text,Original Grade,Final Grade,Meets Target,Rewrites,Needs Fact Check
0,Urinary Tract Infection,"The patient presents with dysuria, urinary frequency, and urgency, consistent with a urinary tract infection.","The patient has pain when urinating, needs to go often, and feels a strong, sudden need to go. These signs point to a urinary tract infection.",15.43,6.27,True,1,True
1,Kidney Stones,"Imaging demonstrates a 5 mm calculus in the left ureter, associated with mild hydronephrosis.",Imaging shows a 5 mm kidney stone in the left ureter. This is causing a mild backup of urine in the kidney.,11.78,4.25,True,1,True
2,Hematuria,The patient exhibits gross hematuria and requires further urological evaluation to determine its etiology.,The patient has blood in the urine that can be seen. The patient needs more tests from a urinary specialist to find the cause.,18.53,4.82,True,2,True
3,Urinary Retention,The patient experiences acute urinary retention requiring bladder decompression via catheterization.,"The patient cannot empty their bladder. A thin, flexible tube is placed to drain the urine.",24.10,4.49,True,1,True
4,Cystoscopy,A cystoscopic examination was performed to evaluate the bladder mucosa for abnormalities.,"A doctor used a thin, lighted tube to look inside the bladder to check for problems.",18.59,6.14,True,1,True



CASES THAT DID NOT MEET TARGET:
All cases met the 8th-grade target!


## 2. Real Urology Dataset Tests

Testing the Readability Agent on 10 samples from the MTSamples urology dataset to see how well it handles real medical reports. Comparing readability scores, rewrite attempts, and any changes in medical meaning.

In [17]:
import ssl
import certifi
import urllib.request
from io import BytesIO

url = "https://raw.githubusercontent.com/socd06/medical-nlp/master/data/mtsamples.csv"

context = ssl.create_default_context(cafile=certifi.where())

with urllib.request.urlopen(url, context=context) as response:
    csv_data = response.read()

df = pd.read_csv(BytesIO(csv_data))

print("Total records:", len(df))
print(df.columns.tolist())

Total records: 4999
['Unnamed: 0', 'description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']


In [18]:
df_urology = df[
    df["medical_specialty"].str.strip() == "Urology"
].copy()

print("Urology records:", len(df_urology))
display(df_urology.head())

Urology records: 158


,Unnamed: 0,description,medical_specialty,sample_name,transcription,keywords
18,18,Fertile male with completed family. Elective male sterilization via bilateral vasectomy.,Urology,Vasectomy - 4,"PROCEDURE: , Elective male sterilization via bilateral vasectomy.,PREOPERATIVE DIAGNOSIS: ,Fertile male with completed family.,POSTOPERATIVE DIAGNOSIS:, Fertile male with completed family.,MEDICATIONS: ,Anesthesia is local with conscious sedation.,COMPLICATIONS: , None.,BLOOD LOSS: , Minimal.,INDICATIONS: ,This 34-year-old gentleman has come to the office requesting sterilization via bilateral vasectomy. I discussed the indications and the need for procedure with the patient in detail, and he has given consent to proceed. He has been given prophylactic antibiotics.,PROCEDURE NOTE: , Once satisfactory sedation have been obtained, the patient was placed in the supine position on the operating table. Genitalia was shaved and then prepped with Betadine scrub and paint solution and were draped sterilely. The procedure itself was started by grasping the right vas deferens in the scrotum, and bringing it up to the level of the skin. The skin was infiltrated with 2% Xylocaine and punctured with a sharp hemostat to identify the vas beneath. The vas was brought out of the incision carefully. A 2-inch segment was isolated, and 1-inch segment was removed. The free ends were cauterized and were tied with 2-0 silk sutures in such a fashion that the ends double back on themselves. After securing hemostasis with a cautery, the ends were allowed to drop back into the incision, which was also cauterized.,Attention was now turned to the left side. The vas was grasped and brought up to the level of the skin. The skin was infiltrated with 2% Xylocaine and punctured with a sharp hemostat to identify the vas beneath. The vas was brought out of the incision carefully. A 2-inch segment was isolated, and 1-inch segment was removed. The free ends were cauterized and tied with 2-0 silk sutures in such a fashion that the ends double back on themselves. After securing hemostasis with the cautery, the ends were allowed to drop back into the incision, which was also cauterized.,Bacitracin ointment was applied as well as dry sterile dressing. The patient was awakened and was returned to Recovery in satisfactory condition.","urology, sterilization, vas, fertile male, bilateral vasectomy, vasectomy, cauterized,"
20,20,Whole body radionuclide bone scan due to prostate cancer.,Urology,Whole Body Radionuclide Bone Scan,"INDICATION:, Prostate Cancer.,TECHNIQUE:, 3.5 hours following the intravenous administration of 26.5 mCi of Technetium 99m MDP, the skeleton was imaged in the anterior and posterior projections.,FINDINGS:, There is a focus of abnormal increased tracer activity overlying the right parietal region of the skull. The uptake in the remainder of the skeleton is within normal limits. The kidneys image normally. There is increased activity in the urinary bladder suggesting possible urinary retention.,CONCLUSION:,1. Focus of abnormal increased tracer activity overlying the right parietal region of the skull. CT scanning of magnetic resonance imaging of the skull and brain could be done for further assessment if it is clinically indicated.,2. There is probably some degree of urinary retention.,","urology, prostate cancer, technetium, whole body, urinary retention, bone scan, radionuclide,"
22,22,Normal vasectomy,Urology,Vasectomy - 1,"DESCRIPTION:, The patient was placed in the supine position and was prepped and draped in the usual manner. The left vas was grasped in between the fingers. The skin and vas were anesthetized with local anesthesia. The vas was grasped with an Allis clamp. Skin was incised and the vas deferens was regrasped with another Allis clamp. The sheath was incised with a scalpel and elevated using the iris scissors and clamps were used to ligate the vas deferens. The portion in between the clamps was excised and the ends of the vas were clamped u

In [21]:
urology_samples = df_urology.dropna(
    subset=["transcription"]
).copy()

def extract_passage(text):
    text = str(text).replace("\n", " ").strip()

    # Look for a clinical history section
    match = re.search(
        r"(?:HISTORY OF PRESENT ILLNESS|BRIEF HISTORY|HISTORY):\s*,?\s*(.*)",
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return None

    passage = match.group(1)

    # Stop when another report section begins
    passage = re.split(
        r"\b(?:PHYSICAL EXAMINATION|ASSESSMENT|PLAN|"
        r"PAST MEDICAL HISTORY|REVIEW OF SYSTEMS|"
        r"OPERATIVE FINDINGS|DESCRIPTION OF PROCEDURE):",
        passage,
        maxsplit=1,
        flags=re.IGNORECASE
    )[0]

    # Take at most the first three complete sentences
    sentences = re.split(r"(?<=[.!?])\s+", passage)
    complete = [s for s in sentences if s.endswith((".", "!", "?"))]

    return " ".join(complete[:3]) if complete else None

urology_samples["test_text"] = (
    urology_samples["transcription"].apply(extract_passage)
)

urology_samples = (
    urology_samples
    .dropna(subset=["test_text"])
    .loc[lambda df: df["test_text"].str.len().between(80, 700)]
    .sample(n=10, random_state=42)
    .reset_index(drop=True)
)

display(urology_samples[["test_text"]])


,test_text
0,"Patient, 37, desires a vasectomy.,PROCEDURE: , Through a midline scrotal incision, the right vas was identified and separated from the surrounding tissues, clamped, transected, and tied off with a 4-0 chromic. No bleeding was identified.,Through the same incision the left side was identified, transected, tied off, and dropped back into the wound. Again no bleeding was noted.,The wound was closed with 4-0 Vicryl times two."
1,"This is a 55-year-old female with a history of stroke, who presents today for followup of frequency and urgency with urge incontinence. This has been progressively worsening, and previously on VESIcare with no improvement. She continues to take Enablex 50 mg and has not noted any improvement of her symptoms."
2,"The patient is an 82-year-old, who was admitted with the history of diabetes, hypertension, hyperlipidemia, coronary artery disease, presented with urinary retention and pneumonia. The patient had hematuria, and unable to void. The patient had a Foley catheter, which was not in the urethra, possibly inflated in the prostatic urethra, which was removed."
3,"The patient is a 71-year-old male whose personal physician, Dr. X identified a change in the patient's PSA from 7/2008 (4.2) to 4/2009 (10.5). The patient underwent a transrectal ultrasound and biopsy and was found to have a Gleason 3+4 for a score of 7, 20% of the tissue removed from the left base."
4,"This is a 10-year-old male who presented to us with his mom with consultation from Craig Connor at Cottonwood with right undescended testis. The patient and mother had seen the testicle in the right hemiscrotum in the past, but the testicle seemed to be sliding. The testis was identified right at the external inguinal ring."
5,"This is a 43-year-old male who presented to us with significant phimosis, difficulty retracting the foreskin. The patient had buried penis with significant obesity issues in the suprapubic area. Options such as watchful waiting, continuation of slowly retracting the skin, applying betamethasone cream, and circumcision were discussed."
6,"A 2-week-old who at this point has otherwise been doing well. He has a relatively unremarkable foreskin. At this point in time, he otherwise seems to be doing reasonably well."
7,This patient is a 66-year-old African-American male who presented to Dr. Y's office with recurrent right inguinal hernia for the second time requesting hernia repair. The procedure was discussed with the patient and the patient opted for laparoscopic repair due to multiple attempts at the open inguinal repair on the right.
8,"The patient presents today for followup, history of erectile dysfunction, last visit started on Cialis 10 mg. He indicates that he has noticed some mild improvement of his symptoms, with no side effect. On this dose, he is having firm erection, able to penetrate, lasting for about 10 or so minutes."
9,This is a 29-year-old African-American male who presents to the Emergency Department today with complaint of a bleeding bump on his penis. The patient states that he has had a large bump on the end of his penis for approximately a year and a half. He states that it has never bled before.


In [22]:
results = []

for i, row in urology_samples.iterrows():
    try:
        result = process_readability(row["test_text"], client)

        results.append({
            "Sample": i + 1,
            "Original Text": row["test_text"],
            "Simplified Text": result["final_text"],
            "Original Grade": result["original_scores"]["flesch_kincaid_grade"],
            "Final Grade": result["final_scores"]["flesch_kincaid_grade"],
            "Meets Target": result["meets_target"],
            "Rewrites": result["rewrite_attempts"],
            "Needs Fact Check": result["requires_faithfulness_check"],
            "Error": None
        })

    except Exception as e:
        results.append({
            "Sample": i + 1,
            "Original Text": row["test_text"],
            "Error": str(e)
        })

df_real = pd.DataFrame(results)

display(df_real)

,Sample,Original Text,Simplified Text,Original Grade,Final Grade,Meets Target,Rewrites,Needs Fact Check,Error
0,1,"Patient, 37, desires a vasectomy.,PROCEDURE: , Through a midline scrotal incision, the right vas was identified and separated from the surrounding tissues, clamped, transected, and tied off with a 4-0 chromic. No bleeding was identified.,Through the same incision the left side was identified, transected, tied off, and dropped back into the wound. Again no bleeding was noted.,The wound was closed with 4-0 Vicryl times two.","Patient, 37, desires a vasectomy.,PROCEDURE: , Through a midline scrotal incision, the right vas was identified and separated from the surrounding tissues, clamped, transected, and tied off with a 4-0 chromic. No bleeding was identified.,Through the same incision the left side was identified, transected, tied off, and dropped back into the wound. Again no bleeding was noted.,The wound was closed with 4-0 Vicryl times two.",7.191875,7.191875,True,0,False,None
1,2,"This is a 55-year-old female with a history of stroke, who presents today for followup of frequency and urgency with urge incontinence. This has been progressively worsening, and previously on VESIcare with no improvement. She continues to take Enablex 50 mg and has not noted any improvement of her symptoms.","This is a 55-year-old female with a history of stroke. She is here today to follow up on a frequent and urgent need to urinate, along with involuntary urine leakage caused by a sudden, strong urge. These symptoms have been getting steadily worse. She previously tried VESIcare with no improvement. She is still taking Enablex 50 mg and has not noticed any improvement in her symptoms.",10.498000,7.436788,True,1,True,None
2,3,"The patient is an 82-year-old, who was admitted with the history of diabetes, hypertension, hyperlipidemia, coronary artery disease, presented with urinary retention and pneumonia. The patient had hematuria, and unable to void. The patient had a Foley catheter, which was not in the urethra, possibly inflated in the prostatic urethra, which was removed.","The patient is an 82-year-old admitted with diabetes, high blood pressure, high cholesterol, and heart disease. The patient came to the hospital with trouble passing urine and a lung infection called pneumonia. The patient also had blood in the urine and could not urinate. A thin tube called a Foley catheter was placed to help drain urine, but it was in the wrong spot. It was likely inflated inside the part of the tube that carries urine through the prostate gland. This catheter was removed.",13.786792,7.287941,True,1,True,None
3,4,"The patient is a 71-year-old male whose personal physician, Dr. X identified a change in the patient's PSA from 7/2008 (4.2) to 4/2009 (10.5). The patient underwent a transrectal ultrasound and biopsy and was found to have a Gleason 3+4 for a score of 7, 20% of the tissue removed from the left base.","The patient is a 71-year-old male whose personal physician, Dr. X identified a change in the patient's PSA from 7/2008 (4.2) to 4/2009 (10.5). The patient underwent a transrectal ultrasound and biopsy and was found to have a Gleason 3+4 for a score of 7, 20% of the tissue removed from the left base.",6.063889,6.063889,True,0,False,None
4,5,"This is a 10-year-old male who presented to us with his mom with consultation from Craig Connor at Cottonwood with right undescended testis. The patient and mother had seen the testicle in the right hemiscrotum in the past, but the testicle seemed to be sliding. The testis was identified right at the external inguinal ring.","This is a 10-year-old male brought in by his mom. He was seen in consultation with Craig Connor at Cottonwood for a right testicle that had not dropped into the proper place. The patient and his mother had seen the testicle in the right side of the scrotum in the past, but it seemed to move back up. The testicle was found right at the opening of the groin canal.",8.938182,6.406

### Results

The Readability Agent met the 8th-grade target on 9 out of 10 real urology samples (90%). Four samples were already below grade 8 and were left unchanged. The agent improved the reading level of the other samples, although one remained above the target after two attempts.

Some medical terms were still difficult to understand even when the text passed the reading-level check. This shows a limitation of using Flesch-Kincaid alone.